In [ ]:
%pip install -U transformers accelerate peft trl datasets sentencepiece bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 27.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 14.1 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [ ]:
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

from trl import SFTTrainer
from peft import prepare_model_for_kbit_training

In [ ]:
#we will use this dataset
#firstly lets load the data
dataset = load_dataset("yahma/alpaca-cleaned")

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [ ]:
dataset

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})

In [ ]:
dataset['train'][0]

{'output': '1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.',
 'input': '',
 'instruction': 'Give three tips for staying healthy.'}

In [ ]:
#lets make teh function for formatting this dataset
dataset=dataset['train'].select(range(20000)) #we will finetune on only on 20k rows


In [ ]:
dataset

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 20000
})

In [ ]:
#lets make the function for the formatting in the prompt template
#in the dataset the instruction can be present or empty so we have handle that also

def formatting_func(example):
    if example["input"]:
        text = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
{example['output']}"""
    else:
        text = f"""### Instruction:
{example['instruction']}

### Response:
{example['output']}"""

    return {"text": text}

In [ ]:
dataset=dataset.map(formatting_func)

Map:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [ ]:
dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 20000
})

In [ ]:
dataset[0]['text'] #working well


'### Instruction:\nGive three tips for staying healthy.\n\n### Response:\n1. Eat a balanced and nutritious diet: Make sure your meals are inclusive of a variety of fruits and vegetables, lean protein, whole grains, and healthy fats. This helps to provide your body with the essential nutrients to function at its best and can help prevent chronic diseases.\n\n2. Engage in regular physical activity: Exercise is crucial for maintaining strong bones, muscles, and cardiovascular health. Aim for at least 150 minutes of moderate aerobic exercise or 75 minutes of vigorous exercise each week.\n\n3. Get enough sleep: Getting enough quality sleep is crucial for physical and mental well-being. It helps to regulate mood, improve cognitive function, and supports healthy growth and immune function. Aim for 7-9 hours of sleep each night.'

In [ ]:
model_name='HuggingFaceTB/SmolLM2-135M'#here we will do full finetuning on the

In [ ]:
#now lets load the full precision LLM and tokenizer
Smol_LLM=AutoModelForCausalLM.from_pretrained(pretrained_model_name_or_path=model_name,device_map='auto')


config.json:   0%|          | 0.00/704 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  269MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

In [ ]:
#now lets load the tokenizer
tokenizer=AutoTokenizer.from_pretrained(pretrained_model_name_or_path=model_name)

tokenizer_config.json:   0%|          | 0.00/3.66k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/801k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/831 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.10M [00:00<?, ?B/s]

In [ ]:
#lets take a inference from it
prompt='Can you tell me what is the capital of france'
input=tokenizer(prompt,return_tensors='pt').to(Smol_LLM.device)



In [ ]:
input

{'input_ids': tensor([[7306,  346, 2505,  549,  732,  314,  260, 3575,  282,  275, 8603]],
       device='cuda:0'), 'attention_mask': tensor([[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]], device='cuda:0')}

In [ ]:
response=Smol_LLM.generate(**input,max_length=100,temperature=0.0001,top_p=0.9,do_sample=True)

In [ ]:
response

tensor([[7306,  346, 2505,  549,  732,  314,  260, 3575,  282,  275, 8603,   47,
          198,  198,  504, 3575,  282, 4649,  314, 7042,   30,  198,  198, 1780,
          314,  260, 3575,  282, 4649,   47,  198,  198,  504, 3575,  282, 4649,
          314, 7042,   30,  198,  198, 1780,  314,  260, 3575,  282, 4649,   47,
          198,  198,  504, 3575,  282, 4649,  314, 7042,   30,  198,  198, 1780,
          314,  260, 3575,  282, 4649,   47,  198,  198,  504, 3575,  282, 4649,
          314, 7042,   30,  198,  198, 1780,  314,  260, 3575,  282, 4649,   47,
          198,  198,  504, 3575,  282, 4649,  314, 7042,   30,  198,  198, 1780,
          314,  260, 3575,  282]], device='cuda:0')

In [ ]:
generated_text=tokenizer.decode(response[0],skip_special_tokens=True)
generated_text

'Can you tell me what is the capital of france?\n\nThe capital of France is Paris.\n\nWhat is the capital of France?\n\nThe capital of France is Paris.\n\nWhat is the capital of France?\n\nThe capital of France is Paris.\n\nWhat is the capital of France?\n\nThe capital of France is Paris.\n\nWhat is the capital of France?\n\nThe capital of France is Paris.\n\nWhat is the capital of'

In [ ]:
#now lets start the finetuning
#we will directly do the finetuning on top of this base model
#lets count the trainable parameters from this LLM
total_params = sum(p.numel() for p in Smol_LLM.parameters())
trainable_params = sum(p.numel() for p in Smol_LLM.parameters() if p.requires_grad)

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

Total Parameters: 134,515,008
Trainable Parameters: 134,515,008


In [ ]:
#so here we are going to retrain each of these parameters of this LLM

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

In [ ]:
#now lets configure the arguments for retraining
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/smol_full_finetuning_on_alpaca",

    num_train_epochs=1,

    per_device_train_batch_size=4,

    gradient_accumulation_steps=2,

    learning_rate=5e-5, #in full finetuning we will use smaller learning rate

    logging_steps=10,

    save_strategy="steps",
    save_steps=25,
    save_total_limit=2,

    fp16=False,
    bf16=True,  #before fp16=True ad bf16=False

    optim="adamw_torch",

    lr_scheduler_type="cosine",

    warmup_ratio=0.03,

    report_to="none",
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [ ]:
#lets create the trainer
trainer = SFTTrainer(
    model=Smol_LLM,
    train_dataset=dataset,
    processing_class=tokenizer,
    args=training_args,
    formatting_func=lambda example: example["text"],
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Applying formatting function to train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Adding EOS to train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/20000 [00:00<?, ? examples/s]

In [1]:
trainer.train()
# trainer.train(resume_from_checkpoint=True) if the training got disconnected then please use this it will restart from where it stopped.

In [1]:
#now lets load these instruction tuned LLM for Inference
%pip install -U transformers accelerate peft trl datasets sentencepiece bitsandbytes


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 885.0/885.0 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 10.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 13.3 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1


In [2]:
import torch

from datasets import load_dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    BitsAndBytesConfig,
)

from peft import (
    LoraConfig,
    get_peft_model,
    TaskType,
)

from trl import SFTTrainer
from peft import prepare_model_for_kbit_training

In [4]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [36]:
model_path='/content/drive/MyDrive/smol_full_finetuning_on_alpaca/checkpoint-2500'
#lets load the llm and tokenizer
fine_tuned_model = AutoModelForCausalLM.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)


Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [6]:
#lets load the dataset
from datasets import load_dataset
dataset=load_dataset('yahma/alpaca-cleaned')

README.md:   0%|          | 0.00/11.6k [00:00<?, ?B/s]

alpaca_data_cleaned.json: reconstructing file:   0%|          |  0.00B / 44.3MB            

alpaca_data_cleaned.json: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/51760 [00:00<?, ? examples/s]

In [8]:
dataset

DatasetDict({
    train: Dataset({
        features: ['output', 'input', 'instruction'],
        num_rows: 51760
    })
})

In [25]:
#we have used the 20k rows of the training dataset now we will evaluate it on the unseen data
testing_dataset=dataset['train'].select(range(51757,51760))

In [26]:
testing_dataset

Dataset({
    features: ['output', 'input', 'instruction'],
    num_rows: 3
})

In [31]:
#lets make the function for the formatting in the prompt template
#in the dataset the instruction can be present or empty so we have handle that also

def formatting_func(example):
    if example["input"]:
        text = f"""### Instruction:
{example['instruction']}

### Input:
{example['input']}

### Response:
"""
    else:
        text = f"""### Instruction:
{example['instruction']}

### Response:
"""

    return {"text": text}

In [28]:
testing_dataset=testing_dataset.map(formatting_func)

Map:   0%|          | 0/3 [00:00<?, ? examples/s]

In [33]:
testing_dataset

Dataset({
    features: ['output', 'input', 'instruction', 'text'],
    num_rows: 3
})

In [40]:
#now lets create the function for the generation of the resoponse
def Response_from_ft_llm(text):


    inputs = tokenizer(text, return_tensors="pt").to(fine_tuned_model.device)

    with torch.no_grad():
        outputs = fine_tuned_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Test it!


for sentence in testing_dataset:
    print(f"Given Instruction: ",sentence['instruction'])
    print(f"Given Input: ",sentence['input'])
    print(f"Refernce:  {sentence['output']}")
    print(f"Response: {Response_from_ft_llm(sentence['text'])}")
    print()

Given Instruction:  You will be given a piece of text about an event that has happened. Your job is to determine if the event could have reasonably happened, based on your knowledge and commonsense. If it could have reasonably happened, output 'True', otherwise output 'False'.
Given Input:  Text: A tree fell over in the wind and caused damage to my car.
Refernce:  True
Response: . The event could have reasonably occurred because there was no wind or other external force acting upon the tree.

Given Instruction:  I will give you a list of steps.  You need to determine if the steps are going forwards or backwards in time by outputting 'Forwards' or 'Backwards'.
Given Input:  Steps: ['She takes out her books', 'The teacher hands back the papers', 'She walks into class', 'The bell rings'].
Refernce:  Backwards
Response: : The steps were taken forward, so they should be completed before the next step is given.

Given Instruction:  Given a piece of text, you need to output whether the statem

In [43]:
#lets also load the base LLM
model_name='HuggingFaceTB/SmolLM2-135M'
base_model=AutoModelForCausalLM.from_pretrained(model_name)

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

In [44]:
#now we will see the response from the base llm
#now lets create the function for the generation of the resoponse
def Response_from_base_llm(text):


    inputs = tokenizer(text, return_tensors="pt").to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=128,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
            repetition_penalty=1.1,
        )

    generated = outputs[0][inputs["input_ids"].shape[1]:]
    return tokenizer.decode(generated, skip_special_tokens=True).strip()

# Test it!


for sentence in testing_dataset:
    print(f"Given Instruction: ",sentence['instruction'])
    print(f"Given Input: ",sentence['input'])
    print(f"Refernce:  {sentence['output']}")
    print(f"Response: {Response_from_base_llm(sentence['text'])}")
    print()

Given Instruction:  You will be given a piece of text about an event that has happened. Your job is to determine if the event could have reasonably happened, based on your knowledge and commonsense. If it could have reasonably happened, output 'True', otherwise output 'False'.
Given Input:  Text: A tree fell over in the wind and caused damage to my car.
Refernce:  True
Response: Explanation:
The event was not likely to happen because I know that trees fall over in the wind.

## Solution 2:

```python
def test_event(self):
    """Test for event."""

    # Create a tree with some data
    self.assertEqual(
        self.tree.get_data(),
        {
            "name": "My name",
            "age": 10,
            "height": 35,
            "weight": 40,
            "gender": "male"
        },
    )

    # Test for events
    self.assertTrue("my name"

Given Instruction:  I will give you a list of steps.  You need to determine if the steps are going forwards or backwards in time by outputting